# GWR/MADD API Test ohne Streamlit

Dieses Notebook testet zuerst den automatischen XML/API-Abruf für eine EGID und parst daraus die Wohnungen. Starte mit `EGID = 64306` und führe die Zellen der Reihe nach aus.

Es werden keine HTML-Tabellen gescraped; der Fokus liegt auf XML/API-Antworten.

In [1]:
# GWR / MADD API Test für eine EGID
# Dieses Notebook ruft die Wohnungsdaten automatisch ab und parst sie in ein DataFrame.

import datetime as dt
import uuid
import xml.etree.ElementTree as ET
from typing import Any, Optional

import pandas as pd
import requests


EGID = 64306  # hier ändern


def safe_num(x, default=None):
    try:
        if x is None:
            return default
        s = str(x).strip().replace(",", ".")
        if s == "":
            return default
        return float(s)
    except Exception:
        return default


def safe_int(x, default=None):
    try:
        v = safe_num(x, None)
        if v is None:
            return default
        return int(round(v))
    except Exception:
        return default


def xml_local_name(tag: str) -> str:
    # Entfernt XML-Namespace, z.B. '{namespace}EGID' -> 'EGID'.
    return tag.split("}", 1)[-1] if "}" in tag else tag


def first_text(parent, *names: str) -> Optional[str]:
    # Sucht rekursiv den ersten Text eines Elementnamens, unabhängig vom Namespace.
    wanted = {n.lower() for n in names}
    for el in parent.iter():
        if xml_local_name(el.tag).lower() in wanted:
            txt = (el.text or "").strip()
            if txt:
                return txt
    return None


def parse_gwr_floor(raw_floor: Any) -> Optional[str]:
    # Floor-Code in lesbares Stockwerk umwandeln.
    # Im MADD/eCH XML für EGID 64306 stehen z.B. 3101, 3102, 3103, 3104.
    # Diese werden im öffentlichen Housing-Stat UI als 1. Stock, 2. Stock usw. angezeigt.
    if raw_floor is None or raw_floor == "":
        return None

    s = str(raw_floor).strip()
    if not s:
        return None

    code = safe_int(s)

    if code is not None:
        if 3101 <= code <= 3199:
            return f"{code - 3100}. Stock"

        # Fallbacks für andere APIs/Codes
        if code == 0:
            return "EG"
        if 1 <= code <= 20:
            return f"{code}. OG"
        if code < 0:
            return f"{abs(code)}. UG"

    return s


def parse_dwellings_from_madd_xml(xml_text: str) -> list[dict]:
    # Parst eCH-0206/MADD XML zu einer Wohnungs-Liste.
    root = ET.fromstring(xml_text)

    dwellings = []
    seen = set()

    for item in root.iter():
        if xml_local_name(item.tag) != "dwellingItem":
            continue

        ewid = safe_int(first_text(item, "EWID"))
        admin_no = first_text(item, "administrativeDwellingNo")
        floor_raw = first_text(item, "floor")

        row = {
            "ewid": ewid,
            "admin_nr": admin_no,
            "floor_raw": safe_int(floor_raw, floor_raw),
            "stockwerk": parse_gwr_floor(floor_raw),
            "rooms": safe_num(first_text(item, "noOfHabitableRooms")),
            "area_sqm": safe_num(first_text(item, "surfaceAreaOfDwelling")),
            "year_built_dwelling": safe_int(first_text(item, "yearOfConstruction")),
            "kitchen": safe_int(first_text(item, "kitchen")),
            "dwelling_status": safe_int(first_text(item, "dwellingStatus")),
        }

        key = ("ewid", ewid) if ewid else (
            "tuple", row["admin_nr"], row["stockwerk"], row["rooms"], row["area_sqm"]
        )

        if key not in seen:
            seen.add(key)
            dwellings.append(row)

    return dwellings


def build_madd_request_xml(egid: int) -> str:
    # Baut eine minimale eCH-0206 maddRequest XML-Anfrage.
    now = dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
    msg_id = str(uuid.uuid4())

    return f'''<?xml version="1.0" encoding="UTF-8"?>
<eCH-0206:maddRequest
    xmlns:eCH-0206="http://www.ech.ch/xmlns/eCH-0206/2"
    xmlns:eCH-0058="http://www.ech.ch/xmlns/eCH-0058/5"
    xmlns:eCH-0129="http://www.ech.ch/xmlns/eCH-0129/5"
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <eCH-0206:requestHeader>
    <eCH-0206:messageId>{msg_id}</eCH-0206:messageId>
    <eCH-0206:businessReferenceId>{msg_id}</eCH-0206:businessReferenceId>
    <eCH-0206:requestingApplication>
      <eCH-0058:manufacturer>Notebook</eCH-0058:manufacturer>
      <eCH-0058:product>GWR-MADD-Test</eCH-0058:product>
      <eCH-0058:productVersion>1.0</eCH-0058:productVersion>
    </eCH-0206:requestingApplication>
    <eCH-0206:requestDate>{now}</eCH-0206:requestDate>
  </eCH-0206:requestHeader>
  <eCH-0206:requestContext>building</eCH-0206:requestContext>
  <eCH-0206:requestQuery>
    <eCH-0206:EGID>{int(egid)}</eCH-0206:EGID>
  </eCH-0206:requestQuery>
</eCH-0206:maddRequest>'''


def looks_like_xml(text: str) -> bool:
    text = (text or "").lstrip()
    return text.startswith("<?xml") or text.startswith("<maddResponse") or "<maddResponse" in text[:500]


def fetch_xml_for_egid(egid: int, timeout: int = 25) -> tuple[str, dict]:
    # Versucht mehrere direkte XML/API-Varianten.
    # Variante 1 ist der direkte XML-Download passend zur Housing-Stat EGID-Seite.
    # Variante 2/3 sind eCH-0206/MADD Webservice-Varianten.
    headers_xml = {
        "Accept": "application/xml,text/xml,*/*",
        "User-Agent": "gwr-madd-api-test-notebook/1.0",
    }

    attempts = []

    # 1) Direkter XML-Endpunkt der öffentlichen EGID-Abfrage
    #    Passt zur HTML-Seite .../egid.html?egid=64306 und dem Button "Als XML herunterladen".
    url = "https://www.housing-stat.ch/de/data/query/egid.xml"
    try:
        r = requests.get(url, params={"egid": int(egid)}, headers=headers_xml, timeout=timeout)
        info = {
            "method": "GET",
            "url": r.url,
            "status": r.status_code,
            "content_type": r.headers.get("Content-Type"),
            "text_start": r.text[:120],
        }
        attempts.append(info)
        if r.status_code == 200 and looks_like_xml(r.text):
            return r.text, {"success": True, "used": info, "attempts": attempts}
    except Exception as exc:
        attempts.append({"method": "GET", "url": url, "exception": f"{type(exc).__name__}: {exc}"})

    # 2) MADD/eCH-0206 als POST mit XML Body
    url = "https://madd.bfs.admin.ch/eCH-0206"
    xml_body = build_madd_request_xml(egid)
    try:
        r = requests.post(
            url,
            data=xml_body.encode("utf-8"),
            headers={
                "Content-Type": "application/xml; charset=utf-8",
                "Accept": "application/xml,text/xml,*/*",
                "User-Agent": "gwr-madd-api-test-notebook/1.0",
            },
            timeout=timeout,
        )
        info = {
            "method": "POST",
            "url": r.url,
            "status": r.status_code,
            "content_type": r.headers.get("Content-Type"),
            "text_start": r.text[:120],
        }
        attempts.append(info)
        if r.status_code == 200 and looks_like_xml(r.text):
            return r.text, {"success": True, "used": info, "attempts": attempts}
    except Exception as exc:
        attempts.append({"method": "POST", "url": url, "exception": f"{type(exc).__name__}: {exc}"})

    # 3) MADD/eCH-0206 als GET mit egid Parameter
    url = "https://madd.bfs.admin.ch/eCH-0206"
    try:
        r = requests.get(url, params={"egid": int(egid)}, headers=headers_xml, timeout=timeout)
        info = {
            "method": "GET",
            "url": r.url,
            "status": r.status_code,
            "content_type": r.headers.get("Content-Type"),
            "text_start": r.text[:120],
        }
        attempts.append(info)
        if r.status_code == 200 and looks_like_xml(r.text):
            return r.text, {"success": True, "used": info, "attempts": attempts}
    except Exception as exc:
        attempts.append({"method": "GET", "url": url, "exception": f"{type(exc).__name__}: {exc}"})

    return "", {"success": False, "attempts": attempts}


xml_text, debug = fetch_xml_for_egid(EGID)

debug

/tmp/ipykernel_19741/55310827.py:122: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


{'success': True,
 'used': {'method': 'POST',
  'url': 'https://madd.bfs.admin.ch/eCH-0206',
  'status': 200,
  'content_type': 'application/xml; charset=UTF-8',
  'text_start': '<?xml version="1.0" encoding="utf-8"?>\n<maddResponse xmlns="http://www.ech.ch/xmlns/eCH-0206/2" xmlns:eCH-0058="http://w'},
 'attempts': [{'method': 'GET',
   'url': 'https://www.housing-stat.ch/de/data/query/egid.xml?egid=64306',
   'status': 404,
   'content_type': 'text/html',
   'text_start': 'ï»¿<!DOCTYPE HTML>\r\n<!--[if gt IE 9 ]><!-->\r\n<html lang="de" class="no-js no-ie">\r\n   <!--<![endif]-->\r\n   <head>\r\n     '},
  {'method': 'POST',
   'url': 'https://madd.bfs.admin.ch/eCH-0206',
   'status': 200,
   'content_type': 'application/xml; charset=UTF-8',
   'text_start': '<?xml version="1.0" encoding="utf-8"?>\n<maddResponse xmlns="http://www.ech.ch/xmlns/eCH-0206/2" xmlns:eCH-0058="http://w'}]}

## XML speichern und kurz prüfen

In [2]:
if not xml_text:
    print("Kein XML erhalten. Debug:")
    for a in debug["attempts"]:
        print(a)
else:
    xml_file = f"egid_{EGID}.xml"
    with open(xml_file, "w", encoding="utf-8") as f:
        f.write(xml_text)

    print(f"XML gespeichert: {xml_file}")
    print(f"Länge: {len(xml_text):,} Zeichen")
    print(xml_text[:500])

XML gespeichert: egid_64306.xml
Länge: 7,559 Zeichen
<?xml version="1.0" encoding="utf-8"?>
<maddResponse xmlns="http://www.ech.ch/xmlns/eCH-0206/2" xmlns:eCH-0058="http://www.ech.ch/xmlns/eCH-0058/5" xmlns:eCH-0129="http://www.ech.ch/xmlns/eCH-0129/5" xmlns:eCH-0007="http://www.ech.ch/xmlns/eCH-0007/6"><status><code>100</code><message>OK</message></status><responseHeader><messageId>036b1ae0-52b5-11f1-be03-0242ac140004</messageId><requestMessageId>9d223a8a-86d8-4f48-8027-a8db9641e8f4</requestMessageId><businessReferenceId>9d223a8a-86d8-4f48-8027-a


## Wohnungen parsen

In [3]:
dwellings = parse_dwellings_from_madd_xml(xml_text) if xml_text else []
df_dwellings = pd.DataFrame(dwellings)

df_dwellings

,ewid,admin_nr,floor_raw,stockwerk,rooms,area_sqm,year_built_dwelling,kitchen,dwelling_status
0,1,101,3101,1. Stock,1.0,60.0,1999,1,3004
1,2,201,3102,2. Stock,3.0,70.0,1999,1,3004
2,3,301,3103,3. Stock,3.0,75.0,1999,1,3004
3,4,401,3104,4. Stock,4.0,85.0,1999,1,3004


## Erwarteter Check für EGID 64306

Für EGID 64306 sollten 4 Wohnungen erscheinen: 60/70/75/85 m² und 1/3/3/4 Zimmer.

In [4]:
if EGID == 64306 and not df_dwellings.empty:
    expected_areas = [60, 70, 75, 85]
    expected_rooms = [1, 3, 3, 4]

    print("Anzahl Wohnungen:", len(df_dwellings))
    print("Flächen:", df_dwellings["area_sqm"].astype(int).tolist())
    print("Zimmer:", df_dwellings["rooms"].astype(int).tolist())

    assert len(df_dwellings) == 4, "Es sollten 4 Wohnungen gefunden werden."
    assert df_dwellings["area_sqm"].astype(int).tolist() == expected_areas, "Flächen stimmen nicht."
    assert df_dwellings["rooms"].astype(int).tolist() == expected_rooms, "Zimmer stimmen nicht."

    print("Check OK.")
else:
    print("Check übersprungen oder keine Daten.")

Anzahl Wohnungen: 4
Flächen: [60, 70, 75, 85]
Zimmer: [1, 3, 3, 4]
Check OK.


## Funktion für deinen Streamlit-Code vorbereiten

Diese Funktion kannst du später in deine App übernehmen.

In [5]:
def load_gwr_dwellings_madd(egid: int) -> list[dict]:
    xml_text, debug = fetch_xml_for_egid(egid)
    if not xml_text:
        raise RuntimeError(f"Keine XML-Antwort erhalten: {debug}")
    return parse_dwellings_from_madd_xml(xml_text)


# Test:
load_gwr_dwellings_madd(EGID)

/tmp/ipykernel_19741/55310827.py:122: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


[{'ewid': 1,
  'admin_nr': '101',
  'floor_raw': 3101,
  'stockwerk': '1. Stock',
  'rooms': 1.0,
  'area_sqm': 60.0,
  'year_built_dwelling': 1999,
  'kitchen': 1,
  'dwelling_status': 3004},
 {'ewid': 2,
  'admin_nr': '201',
  'floor_raw': 3102,
  'stockwerk': '2. Stock',
  'rooms': 3.0,
  'area_sqm': 70.0,
  'year_built_dwelling': 1999,
  'kitchen': 1,
  'dwelling_status': 3004},
 {'ewid': 3,
  'admin_nr': '301',
  'floor_raw': 3103,
  'stockwerk': '3. Stock',
  'rooms': 3.0,
  'area_sqm': 75.0,
  'year_built_dwelling': 1999,
  'kitchen': 1,
  'dwelling_status': 3004},
 {'ewid': 4,
  'admin_nr': '401',
  'floor_raw': 3104,
  'stockwerk': '4. Stock',
  'rooms': 4.0,
  'area_sqm': 85.0,
  'year_built_dwelling': 1999,
  'kitchen': 1,
  'dwelling_status': 3004}]